# CropCare AI MobileNetV2 training (Google Colab)

This notebook trains the configured 15-class classifier using only the existing training and validation manifests. It preserves the project class mapping and training-only class weights. The test split is not read, loaded, or evaluated anywhere in this workflow.

**GPU required:** the runtime check stops execution if Colab has not attached a GPU. Select **Runtime > Change runtime type > T4 GPU** (or another available GPU) and rerun the setup cells. Training starts only when you explicitly run the `model.fit` cell.

## Colab and Drive setup

Run the Drive setup code cell immediately below before running any validation cells. It mounts Google Drive, initializes every project path, and checks that required directories and files exist. Copy the project data and config into Drive with this layout:

```text
MyDrive/CropCare-AI/
├── dataset/
│   ├── processed/
│   │   ├── train_manifest.csv
│   │   ├── validation_manifest.csv
│   │   ├── class_mapping.json
│   │   └── class_weights.json
│   └── raw/PlantVillage/PlantVillage/<class>/<image>
└── scripts/training_config.json
```

The existing manifest paths may contain Windows paths or the earlier project-relative path. The validation cell resolves each row using its unchanged class-folder name and filename under the actual Drive raw directory. Do not regenerate or edit the manifests.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
PROJECT_DIR = Path('/content/drive/MyDrive/CropCare-AI')
RAW_DATA_DIR = Path('/content/drive/MyDrive/CropCare-AI/dataset/raw/PlantVillage/PlantVillage')
PROCESSED_DIR = Path('/content/drive/MyDrive/CropCare-AI/dataset/processed')
CONFIG_PATH = Path('/content/drive/MyDrive/CropCare-AI/scripts/training_config.json')
TRAIN_MANIFEST_PATH = Path('/content/drive/MyDrive/CropCare-AI/dataset/processed/train_manifest.csv')
VALIDATION_MANIFEST_PATH = Path('/content/drive/MyDrive/CropCare-AI/dataset/processed/validation_manifest.csv')
CLASS_MAPPING_PATH = Path('/content/drive/MyDrive/CropCare-AI/dataset/processed/class_mapping.json')
CLASS_WEIGHTS_PATH = Path('/content/drive/MyDrive/CropCare-AI/dataset/processed/class_weights.json')
SCRIPTS_DIR = Path('/content/drive/MyDrive/CropCare-AI/scripts')
OUTPUT_DIR = PROJECT_DIR / 'model' / 'training'

required_directories = [PROJECT_DIR, RAW_DATA_DIR, PROCESSED_DIR, SCRIPTS_DIR]
missing_directories = [str(path) for path in required_directories if not path.is_dir()]
if missing_directories:
    raise FileNotFoundError('Required Google Drive directories are missing:\n' + '\n'.join(missing_directories))
required_files = [
    TRAIN_MANIFEST_PATH,
    VALIDATION_MANIFEST_PATH,
    CLASS_MAPPING_PATH,
    CLASS_WEIGHTS_PATH,
    CONFIG_PATH,
]
missing_files = [str(path) for path in required_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError('Required project files are missing:\n' + '\n'.join(missing_files))
print(f'Project directory: {PROJECT_DIR}')
print(f'Raw dataset directory: {RAW_DATA_DIR}')
print(f'Train manifest: {TRAIN_MANIFEST_PATH}')
print(f'Validation manifest: {VALIDATION_MANIFEST_PATH}')
print(f'Class mapping: {CLASS_MAPPING_PATH}')
print(f'Class weights: {CLASS_WEIGHTS_PATH}')
print(f'Training config: {CONFIG_PATH}')
print(f'Output directory: {OUTPUT_DIR}')

## Load and validate training inputs

This preflight opens only the training and validation manifests. It verifies all 15 class folders, compares per-class raw image counts with the original local dataset inventory, resolves every manifest filename under the actual Drive raw directory, and checks for duplicate hashes across both splits. It prints resolved counts and stops if any image is missing or any source/Drive count differs. It does not open the test manifest. Run the Drive setup cell above first; this cell checks that its variables were initialized before use.

In [ ]:
required_setup_variables = (
    'PROJECT_DIR', 'PROCESSED_DIR', 'RAW_DATA_DIR', 'CONFIG_PATH',
    'TRAIN_MANIFEST_PATH', 'VALIDATION_MANIFEST_PATH',
    'CLASS_MAPPING_PATH', 'CLASS_WEIGHTS_PATH',
)
uninitialized_variables = [name for name in required_setup_variables if name not in globals()]
if uninitialized_variables:
    raise RuntimeError(
        'Run the Google Drive setup cell above before this validation cell. '
        f'Not initialized: {uninitialized_variables}'
    )

import csv
import json
import math
import random
from collections import Counter
from pathlib import PureWindowsPath

import numpy as np

with CONFIG_PATH.open(encoding='utf-8') as file:
    config = json.load(file)
with CLASS_MAPPING_PATH.open(encoding='utf-8') as file:
    mapping_data = json.load(file)
class_to_index = mapping_data.get('class_to_index')
index_to_class = mapping_data.get('index_to_class')
if not isinstance(class_to_index, dict) or not isinstance(index_to_class, dict):
    raise ValueError('class_mapping.json must contain class_to_index and index_to_class.')
if len(class_to_index) != 15 or len(index_to_class) != 15:
    raise ValueError('The saved class mapping must define exactly 15 classes.')
classes = []
for index in range(15):
    class_name = index_to_class.get(str(index))
    if class_name is None or class_to_index.get(class_name) != index:
        raise ValueError('The saved class mapping must be contiguous and reversible.')
    classes.append(class_name)

# Image-file counts measured from the original local PlantVillage class folders.
ORIGINAL_IMAGE_COUNTS = {
    'Pepper__bell___Bacterial_spot': 997,
    'Pepper__bell___healthy': 1478,
    'Potato___Early_blight': 1000,
    'Potato___healthy': 152,
    'Potato___Late_blight': 1000,
    'Tomato__Target_Spot': 1404,
    'Tomato__Tomato_mosaic_virus': 373,
    'Tomato__Tomato_YellowLeaf__Curl_Virus': 3208,
    'Tomato_Bacterial_spot': 2127,
    'Tomato_Early_blight': 1000,
    'Tomato_healthy': 1591,
    'Tomato_Late_blight': 1909,
    'Tomato_Leaf_Mold': 952,
    'Tomato_Septoria_leaf_spot': 1771,
    'Tomato_Spider_mites_Two_spotted_spider_mite': 1676,
}
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.webp'}
drive_class_folders = {path.name for path in RAW_DATA_DIR.iterdir() if path.is_dir()}
if drive_class_folders != set(classes):
    missing_classes = sorted(set(classes) - drive_class_folders)
    extra_folders = sorted(drive_class_folders - set(classes))
    raise ValueError(f'Drive class folders do not match the saved 15-class mapping. Missing: {missing_classes}; extra: {extra_folders}')
if set(ORIGINAL_IMAGE_COUNTS) != set(classes):
    raise ValueError('Original image-count baseline does not match the saved class mapping.')
drive_image_counts = {
    class_name: sum(
        path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
        for path in (RAW_DATA_DIR / class_name).rglob('*')
    )
    for class_name in classes
}
original_total = sum(ORIGINAL_IMAGE_COUNTS.values())
drive_total = sum(drive_image_counts.values())
count_differences = {
    name: (ORIGINAL_IMAGE_COUNTS[name], drive_image_counts[name])
    for name in classes
    if ORIGINAL_IMAGE_COUNTS[name] != drive_image_counts[name]
}
print(f'Original dataset image files: {original_total}')
print(f'Drive copy image files: {drive_total}')
if count_differences:
    print('Per-class image count differences (original, Drive):', count_differences)
    raise ValueError('Drive raw-image counts differ from the original dataset inventory.')
print('Drive copy image counts match the original dataset in all 15 classes.')

expected_config = {
    'class_count': 15,
    'image_size': [224, 224],
    'pretrained_weights': 'imagenet',
}
for key, expected in expected_config.items():
    if config.get(key) != expected:
        raise ValueError(f'Unexpected training config {key}: {config.get(key)!r}')
for key in ('seed', 'epochs', 'batch_size', 'early_stopping_patience', 'reduce_lr_patience'):
    if not isinstance(config.get(key), int) or config[key] <= 0:
        raise ValueError(f'{key} must be a positive integer.')

required_columns = {'image_path', 'class_name', 'class_index', 'split', 'image_hash'}
missing_manifest_images = []
def load_manifest(manifest_path, expected_split):
    image_paths, labels, hashes = [], [], []
    row_count = 0
    with manifest_path.open('r', newline='', encoding='utf-8-sig') as file:
        reader = csv.DictReader(file)
        if reader.fieldnames is None or not required_columns.issubset(reader.fieldnames):
            raise ValueError(f'{manifest_path.name} is missing required columns: {sorted(required_columns)}')
        for line_number, row in enumerate(reader, start=2):
            row_count += 1
            class_name = row['class_name']
            if class_name not in class_to_index:
                raise ValueError(f'{manifest_path.name}:{line_number} contains an unknown class.')
            class_index = int(row['class_index'])
            if class_index != class_to_index[class_name] or row['split'] != expected_split:
                raise ValueError(f'{manifest_path.name}:{line_number} disagrees with the saved mapping/split.')
            image_hash = row['image_hash']
            if not image_hash:
                raise ValueError(f'{manifest_path.name}:{line_number} has no image hash.')
            path_parts = PureWindowsPath(row['image_path']).parts
            filename = PureWindowsPath(row['image_path']).name
            if class_name not in path_parts[:-1] or not filename:
                raise ValueError(f'{manifest_path.name}:{line_number} does not include its class folder and filename.')
            image_path = (RAW_DATA_DIR / class_name / filename).resolve()
            if not image_path.is_relative_to(RAW_DATA_DIR) or not image_path.is_file():
                missing_manifest_images.append((manifest_path.name, line_number, str(image_path), row['image_path']))
                hashes.append(image_hash)
                continue
            image_paths.append(str(image_path))
            labels.append(class_index)
            hashes.append(image_hash)
    return image_paths, np.asarray(labels, dtype=np.int32), hashes, row_count

train_paths, train_labels, train_hashes, train_rows = load_manifest(TRAIN_MANIFEST_PATH, 'train')
validation_paths, validation_labels, validation_hashes, validation_rows = load_manifest(VALIDATION_MANIFEST_PATH, 'validation')
expected_manifest_rows = {'train': 14436, 'validation': 3094}
manifest_row_counts = {'train': train_rows, 'validation': validation_rows}
print(f'Manifest row counts (expected): {manifest_row_counts} (expected {expected_manifest_rows})')
if manifest_row_counts != expected_manifest_rows:
    raise ValueError(f'Manifest row counts differ from the verified split counts: {manifest_row_counts}')
print(f'Training images resolved: {len(train_paths)}/{train_rows}')
print(f'Validation images resolved: {len(validation_paths)}/{validation_rows}')
print(f'Total manifest images resolved: {len(train_paths) + len(validation_paths)}/{train_rows + validation_rows}')
if missing_manifest_images:
    print(f'Missing manifest images: {len(missing_manifest_images)}')
    for manifest_name, line_number, resolved_path, source_path in missing_manifest_images[:25]:
        print(f'  {manifest_name}:{line_number}: {source_path} -> {resolved_path}')
    raise FileNotFoundError(f'{len(missing_manifest_images)} manifest image(s) are missing from Google Drive; stop here and do not train.')
if not train_paths or set(train_labels) != set(range(15)):
    raise ValueError('The training manifest is empty or does not contain all 15 classes.')
if not validation_paths or set(validation_labels) != set(range(15)):
    raise ValueError('The validation manifest is empty or does not contain all 15 classes.')
all_hashes = train_hashes + validation_hashes
duplicate_hashes = sorted(hash_value for hash_value, count in Counter(all_hashes).items() if count > 1)
if duplicate_hashes:
    print(f'Duplicate hashes across training and validation: {len(duplicate_hashes)}')
    raise ValueError('Duplicate image hashes were found across the training/validation manifests.')
print(f'Unique hashes across both manifests: {len(all_hashes)}; duplicates: 0')
with CLASS_WEIGHTS_PATH.open(encoding='utf-8') as file:
    weights_data = json.load(file)
if weights_data.get('method') != 'sklearn.utils.class_weight.compute_class_weight' or weights_data.get('class_weight') != 'balanced':
    raise ValueError('Expected the saved balanced weights generated by sklearn.')
actual_training_counts = Counter(classes[index] for index in train_labels)
if weights_data.get('training_counts') != dict(actual_training_counts):
    raise ValueError('Saved class-weight counts do not match the training manifest.')
saved_weights = weights_data.get('weights', {})
if set(saved_weights) != set(classes):
    raise ValueError('Saved class weights do not match the class mapping.')
class_weights = {class_to_index[name]: float(saved_weights[name]) for name in classes}
if any(not math.isfinite(weight) or weight <= 0 for weight in class_weights.values()):
    raise ValueError('All class weights must be finite positive values.')

print(f'Classes: {len(classes)}')
print('Class mapping:', class_to_index)
print(f'Training images resolved: {len(train_paths)}')
print(f'Validation images resolved: {len(validation_paths)}')
print(f'Configured epochs/batch size: {config["epochs"]}/{config["batch_size"]}')
print('Class mapping and training-only class weights: validated')

In [ ]:
import tensorflow as tf
import keras

gpus = tf.config.list_physical_devices('GPU')
print(f'TensorFlow: {tf.__version__}')
print(f'Keras: {keras.__version__}')
print(f'GPU devices: {gpus}')
if not gpus:
    raise RuntimeError(
        'No GPU is attached to this Colab runtime. Select Runtime > Change runtime type > '
        'a GPU accelerator, then rerun the setup cells. Training was not started.'
    )
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
print('GPU runtime check: PASS')

## Build input pipeline and frozen ImageNet MobileNetV2

In [ ]:
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras import Model, layers

random.seed(config['seed'])
np.random.seed(config['seed'])
tf.keras.utils.set_random_seed(config['seed'])

def make_dataset(image_paths, labels, training):
    dataset = tf.data.Dataset.from_tensor_slices((image_paths, labels))
    if training:
        dataset = dataset.shuffle(
            buffer_size=len(image_paths), seed=config['seed'], reshuffle_each_iteration=True
        )
    def load_image(image_path, label):
        image_bytes = tf.io.read_file(image_path)
        image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
        image.set_shape([None, None, 3])
        image = tf.image.resize(image, config['image_size'])
        image = preprocess_input(image)
        image.set_shape([224, 224, 3])
        return image, label
    dataset = dataset.map(load_image, num_parallel_calls=tf.data.AUTOTUNE, deterministic=True)
    return dataset.batch(config['batch_size']).prefetch(tf.data.AUTOTUNE)

train_dataset = make_dataset(train_paths, train_labels, training=True)
validation_dataset = make_dataset(validation_paths, validation_labels, training=False)

base_model = MobileNetV2(
    input_shape=(224, 224, 3), include_top=False, weights=config['pretrained_weights']
)
base_model.trainable = False
inputs = layers.Input(shape=(224, 224, 3), name='image')
features = base_model(inputs, training=False)
features = layers.GlobalAveragePooling2D(name='global_average_pooling')(features)
features = layers.Dropout(config['dropout_rate'], name='dropout')(features)
outputs = layers.Dense(len(classes), activation='softmax', name='predictions')(features)
model = Model(inputs=inputs, outputs=outputs, name='cropcare_mobilenetv2')
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=config['learning_rate']),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
print(f'Frozen base: {base_model.name}; classes: {len(classes)}')
model.summary()

## Train on training data and validate each epoch

Running the next cell starts training. It uses configured epochs and batch size, the saved class weights, and the three configured callbacks. Accuracy, loss, validation accuracy, and validation loss are captured for every completed epoch. The test split is not used.

In [ ]:
import time

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
best_model_path = OUTPUT_DIR / 'best_model.keras'
callbacks = [
    EarlyStopping(
        monitor='val_loss', patience=config['early_stopping_patience'],
        restore_best_weights=True, verbose=1,
    ),
    ModelCheckpoint(
        filepath=str(best_model_path), monitor='val_loss', save_best_only=True,
        mode='min', verbose=1,
    ),
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.2, patience=config['reduce_lr_patience'],
        min_lr=config['minimum_learning_rate'], mode='min', verbose=1,
    ),
]
start_time = time.perf_counter()
history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=config['epochs'],
    class_weight=class_weights,
    callbacks=callbacks,
)
training_duration_seconds = time.perf_counter() - start_time
print(f'Completed epochs: {len(history.epoch)}')
print(f'Training duration (seconds): {training_duration_seconds:.2f}')

## Save best model and training artifacts

In [ ]:
import shutil

if not best_model_path.is_file():
    raise FileNotFoundError(f'ModelCheckpoint did not create {best_model_path}')
class_mapping_path = OUTPUT_DIR / 'class_mapping.json'
shutil.copy2(PROCESSED_DIR / 'class_mapping.json', class_mapping_path)
history_json_path = OUTPUT_DIR / 'training_history.json'
history_csv_path = OUTPUT_DIR / 'training_history.csv'
history_data = {
    'config': config,
    'classes': classes,
    'epochs_completed': len(history.epoch),
    'training_duration_seconds': training_duration_seconds,
    'history': {key: [float(value) for value in values] for key, values in history.history.items()},
    'test_evaluation': 'not performed',
}
history_json_path.write_text(json.dumps(history_data, indent=2) + '\n', encoding='utf-8')
with history_csv_path.open('w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    metric_names = list(history_data['history'])
    writer.writerow(['epoch', *metric_names])
    for epoch_index in range(len(history.epoch)):
        writer.writerow([epoch_index + 1, *(history_data['history'][name][epoch_index] for name in metric_names)])
shutil.copy2(CONFIG_PATH, OUTPUT_DIR / 'training_config.json')

val_losses = history.history['val_loss']
best_loss_epoch_index = int(np.argmin(val_losses))
best_accuracy_epoch_index = int(np.argmax(history.history['val_accuracy']))
best_validation_loss = float(val_losses[best_loss_epoch_index])
best_validation_accuracy = float(history.history['val_accuracy'][best_accuracy_epoch_index])
print(f'Completed epochs: {len(history.epoch)}')
print(f'Best validation accuracy: {best_validation_accuracy:.6f} (epoch {best_accuracy_epoch_index + 1})')
print(f'Best validation loss: {best_validation_loss:.6f} (epoch {best_loss_epoch_index + 1})')
print(f'Training duration (seconds): {training_duration_seconds:.2f}')
print(f'Best model: {best_model_path}')
print(f'Class mapping: {class_mapping_path}')
print(f'History JSON/CSV: {history_json_path} ; {history_csv_path}')

## Export artifacts for local download

Run this cell after a successful training and save step. It packages the model, class mapping, config, and both history files into one ZIP, then starts a browser download. The ZIP is also retained in the Drive project’s `model/` folder.

In [ ]:
import zipfile
from google.colab import files

artifact_paths = [
    best_model_path,
    class_mapping_path,
    history_json_path,
    history_csv_path,
    OUTPUT_DIR / 'training_config.json',
]
missing_artifacts = [str(path) for path in artifact_paths if not path.is_file()]
if missing_artifacts:
    raise FileNotFoundError('Cannot export; missing artifacts:\n' + '\n'.join(missing_artifacts))
zip_path = PROJECT_DIR / 'model' / 'cropcare_mobilenetv2_artifacts.zip'
zip_path.parent.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in artifact_paths:
        archive.write(path, arcname=path.name)
print(f'Created archive: {zip_path}')
files.download(str(zip_path))